# Sentiment analysis of ayam gepuk reviews

This notebook summarises the **existing manual sentiment annotations**; it does not train a classifier or infer labels from ratings. The unit of analysis is one complete review. All language groups are included.

Run all cells from the repository root or `Notebooks` directory. Dependencies: `pandas`, `openpyxl`, `matplotlib` (install in your notebook environment if needed).

Neutral follows the team's recorded annotations, including mixed evaluations. No labels are changed here. The results are descriptive, not causal, and are not a representative estimate of all customers.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "Data" / "CPC 353 Dataset.xlsx").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Run from the repository root or a directory inside it.")
INPUT_FILE = PROJECT_DIR / "Data" / "CPC 353 Dataset.xlsx"
TABLE_DIR = PROJECT_DIR / "Outputs" / "Tables"
FIGURE_DIR = PROJECT_DIR / "Outputs" / "Figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENTS = ["Positive", "Negative", "Neutral"]
SWITCH_TYPES = ["Intra-sentential", "Inter-sentential", "Tag Switching", "N/A"]
COLORS = ["#287D8E", "#B65044", "#8B8D91"]
print("Input:", INPUT_FILE.name)
print("SHA-256:", hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest())


Input: CPC 353 Dataset.xlsx
SHA-256: bd587042d3bd3d1c514c9bf7dfa41bbbe92de58d676b666aa9fb12afb3f54dae


## Load and validate

Literal `N/A` means no switching and is retained as a category. Empty or unknown sentiment labels and duplicate review IDs stop analysis rather than silently changing the denominator. Only completely empty worksheet rows are removed.


In [2]:
df = pd.read_excel(INPUT_FILE, sheet_name="Self-Collected Data", header=3,
                   usecols="A:M", keep_default_na=False)
df = df.loc[df.astype(str).apply(lambda s: s.str.strip().ne("")).any(axis=1)].copy()
required = ["Review ID", "Review Text", "Language Classification", "Code-Switching Type", "Sentiment"]
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
for column in required:
    df[column] = df[column].astype(str).str.strip()

def validate_reviews(data):
    if data.empty:
        raise ValueError("No reviews available.")
    if data["Review ID"].eq("").any() or data["Review ID"].duplicated().any():
        raise ValueError("Review IDs must be present and unique.")
    if data["Review Text"].eq("").any():
        raise ValueError("Review text is missing.")
    for column, allowed in [("Sentiment", SENTIMENTS),
                            ("Code-Switching Type", SWITCH_TYPES),
                            ("Language Classification", ["English", "Malay", "Code-switched"])]:
        invalid = data.loc[~data[column].isin(allowed), ["Review ID", column]]
        if not invalid.empty:
            raise ValueError(f"Resolve invalid or missing {column} labels:\n{invalid}")
    inconsistent = data["Language Classification"].eq("Code-switched") == data["Code-Switching Type"].eq("N/A")
    if inconsistent.any():
        raise ValueError("Language classification and switching type disagree.")

validate_reviews(df)
print(f"Validated {len(df)} reviews; all sentiment labels are present.")


Validated 197 reviews; all sentiment labels are present.


## Table 1: overall sentiment

Percentage = count / total reviews × 100. Rounding may make displayed percentages sum to 100.01%.


In [3]:
overall = df["Sentiment"].value_counts().reindex(SENTIMENTS, fill_value=0).rename("review_count").to_frame()
overall.index.name = "sentiment"
overall["percentage"] = (overall["review_count"] / len(df) * 100).round(2)
overall["denominator"] = len(df)
display(overall)


           review_count  percentage  denominator
sentiment                                       
Positive            115       58.38          197
Negative             56       28.43          197
Neutral              26       13.20          197


## Table 2: sentiment by primary switching type

Each percentage uses that switching group's total, not all reviews. N/A is the non-switching comparison group. Unobserved switching categories are omitted; absent sentiments within an observed group remain zero. Reviews containing both switching types retain their assigned primary label.


In [4]:
observed_types = [t for t in SWITCH_TYPES if t in set(df["Code-Switching Type"])]
counts = pd.crosstab(df["Code-Switching Type"], df["Sentiment"]).reindex(index=observed_types, columns=SENTIMENTS, fill_value=0)
counts.index.name = "switching_type"
totals = counts.sum(axis=1)
percentages = counts.div(totals, axis=0).mul(100).round(2)
report = pd.DataFrame(index=counts.index)
for sentiment in SENTIMENTS:
    report[sentiment + " n (%)"] = [f"{n} ({p:.2f}%)" for n, p in zip(counts[sentiment], percentages[sentiment])]
report["Total"] = totals
display(report)
assert counts.to_numpy().sum() == len(df)
assert counts.sum(axis=0).equals(overall["review_count"].rename_axis("Sentiment"))
assert (percentages.sum(axis=1) - 100).abs().lt(0.02).all()


                 Positive n (%) Negative n (%) Neutral n (%)  Total
switching_type                                                     
Intra-sentential    80 (57.55%)    43 (30.94%)   16 (11.51%)    139
Inter-sentential    5 (100.00%)      0 (0.00%)     0 (0.00%)      5
N/A                 30 (56.60%)    13 (24.53%)   10 (18.87%)     53


## Export reproducible tables

Numeric counts and percentages are exported separately for reuse, alongside the formatted report table. Only this notebook's output files are replaced on rerun.


In [5]:
overall.to_csv(TABLE_DIR / "sentiment_distribution.csv", encoding="utf-8-sig")
counts.assign(Total=totals).to_csv(TABLE_DIR / "sentiment_by_switching_counts.csv", encoding="utf-8-sig")
percentages.assign(denominator=totals).to_csv(TABLE_DIR / "sentiment_by_switching_percentages.csv", encoding="utf-8-sig")
report.to_csv(TABLE_DIR / "sentiment_by_switching_report.csv", encoding="utf-8-sig")
print("Exported four sentiment tables.")


Exported four sentiment tables.


## Figures

The first chart shows review counts. The second compares proportions and displays group sizes; the inter-sentential group is very small, so its percentage must be interpreted cautiously.


In [6]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(SENTIMENTS, overall["review_count"], color=COLORS)
for bar, n, p in zip(bars, overall["review_count"], overall["percentage"]):
    ax.text(bar.get_x() + bar.get_width()/2, n + len(df)*0.015, f"{n} ({p:.2f}%)", ha="center")
ax.set_ylim(0, max(overall["review_count"])*1.22)
ax.set_ylabel("Number of reviews")
ax.set_title(f"Overall sentiment (n = {len(df)})")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sentiment_distribution.png", dpi=300)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
left = pd.Series(0.0, index=counts.index)
for sentiment, color in zip(SENTIMENTS, COLORS):
    widths = counts[sentiment] / totals * 100
    ax.barh(counts.index, widths, left=left, label=sentiment, color=color)
    for i, (start, width) in enumerate(zip(left, widths)):
        if width >= 8:
            ax.text(start + width/2, i, f"{width:.1f}%", ha="center", va="center", color="white")
    left += widths
ax.set_yticks(range(len(totals)), [f"{t} (n={n})" for t, n in totals.items()])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Percentage within switching group")
ax.set_title("Sentiment by primary switching type")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20), ncol=3, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sentiment_by_switching.png", dpi=300)
plt.show()
plt.close(fig)


## Interpretation and handoff

Use the computed tables to describe the sample, then select passages from the concordance output to explain negation, praise, complaints, contrast and expectations. A keyword occurrence is not a sentiment label or automatically a switching event. Passage sentiment may differ from the overall review sentiment.

In the initial 197-review snapshot, intra-sentential and non-switching reviews have similar positive shares. All five inter-sentential reviews are positive, but the small group does not justify a broad claim about that switching type. These observations do not establish causation.

The switching-distribution percentages use only code-switched reviews as their denominator; this notebook's within-group percentages answer a different question.
